## Numpy

In [1]:
import numpy as np

### Zadanie 1.

Stwórz losową tablicę 10,000 liczb, następnie do każdego elementu dodaj 1. Następnie napisz dokładnie tę samą funkcję używając `numpy` i jego tablic. Porównaj czas działania swoich rozwiązań.

In [2]:
import random
from time import perf_counter


def add_one_loop(values):
    return [value + 1 for value in values]


def add_one_numpy(values):
    return np.asarray(values) + 1


values = [random.random() for _ in range(10_000)]
start = perf_counter()
loop_result = add_one_loop(values)
loop_time = perf_counter() - start
start = perf_counter()
numpy_result = add_one_numpy(values)
numpy_time = perf_counter() - start
assert np.allclose(loop_result, numpy_result)
print(f"loop: {loop_time:.6f}s; NumPy: {numpy_time:.6f}s")

loop: 0.000180s; NumPy: 0.000244s


### Zadanie 2.

Napisz funkcje obliczajaca $\sin(x) + \cos(x)$ dla kazdego elementu tablicy 10,000 losowych liczb używając pętli oraz używając numpy, porownaj ich szybkość podobnie jak w poprzednim zadaniu.

In [3]:
from math import sin, cos
from time import perf_counter


def trig_loop(values):
    return [sin(value) + cos(value) for value in values]


def trig_numpy(values):
    values = np.asarray(values)
    return np.sin(values) + np.cos(values)


values = np.random.random(10_000)
start = perf_counter()
loop_result = trig_loop(values)
loop_time = perf_counter() - start
start = perf_counter()
numpy_result = trig_numpy(values)
numpy_time = perf_counter() - start
assert np.allclose(loop_result, numpy_result)
print(f"loop: {loop_time:.6f}s; NumPy: {numpy_time:.6f}s")

loop: 0.000580s; NumPy: 0.000109s


### Zadanie 3.

Napisz wydajniejszą forme następujących funkcji. Dopisz do każdej test potwierdzający szybkość zaproponowanego rozwiązania.

In [4]:
def cube_sum(x):
    """Zwraca sume szescianow elementow"""
    result = 0
    for i in range(len(x)):
        result += x[i] ** 3
    return result


def almost_variance(x):
    """Oblicza 1/n * SUM (x_i - mean(x))^4"""
    m = sum(x) / len(x)
    result = 0
    for i in range(len(x)):
        result += (x[i] - m) ** 4
    result /= len(x)
    return result


def cube_sum_numpy(x):
    return np.sum(np.asarray(x) ** 3)


def almost_variance_numpy(x):
    values = np.asarray(x, dtype=float)
    return np.mean((values - values.mean()) ** 4)


values = np.random.default_rng(0).random(100_000)
start = perf_counter()
old_cube_sum = cube_sum(values)
old_cube_time = perf_counter() - start
start = perf_counter()
new_cube_sum = cube_sum_numpy(values)
new_cube_time = perf_counter() - start
start = perf_counter()
old_variance = almost_variance(values)
old_variance_time = perf_counter() - start
start = perf_counter()
new_variance = almost_variance_numpy(values)
new_variance_time = perf_counter() - start
np.testing.assert_allclose([old_cube_sum, old_variance], [new_cube_sum, new_variance])
assert new_cube_time < old_cube_time and new_variance_time < old_variance_time
print(f"cube: {old_cube_time:.6f}s -> {new_cube_time:.6f}s")
print(f"variance: {old_variance_time:.6f}s -> {new_variance_time:.6f}s")

cube: 0.008581s -> 0.000609s
variance: 0.014181s -> 0.001212s


### Zadanie 4.

Wykorzystujac `broadcasting` (patrz na rysunek poniżej) stwórz tabliczkę mnożenia tzn. tablice $A$ taka, ze $$A_{ij} = i \cdot j\;\;\;\;\; \forall i,j \in \{1,\dots,10\}$$

Czy potrafisz wykonać to jednym poleceniem, nie używajac przy tym petli/comprehensions?

<img src="http://www.astroml.org/_images/fig_broadcast_visual_1.png">

In [5]:
numbers = np.arange(1, 11)
multiplication_table = numbers[:, None] * numbers[None, :]
assert multiplication_table.shape == (10, 10)
assert multiplication_table[0, 0] == 1 and multiplication_table[-1, -1] == 100
multiplication_table

array([[  1,   2,   3,   4,   5,   6,   7,   8,   9,  10],
       [  2,   4,   6,   8,  10,  12,  14,  16,  18,  20],
       [  3,   6,   9,  12,  15,  18,  21,  24,  27,  30],
       [  4,   8,  12,  16,  20,  24,  28,  32,  36,  40],
       [  5,  10,  15,  20,  25,  30,  35,  40,  45,  50],
       [  6,  12,  18,  24,  30,  36,  42,  48,  54,  60],
       [  7,  14,  21,  28,  35,  42,  49,  56,  63,  70],
       [  8,  16,  24,  32,  40,  48,  56,  64,  72,  80],
       [  9,  18,  27,  36,  45,  54,  63,  72,  81,  90],
       [ 10,  20,  30,  40,  50,  60,  70,  80,  90, 100]])

### Zad 5.

Wygeneruj losową tablice `10 x 5`, która będzie reprezentować `10` punktów w `5` wymiarowej przestrzeni. Napisz funkcje, która policzy odleglość euklidesową pomiędzy każdą parą punktów, czy potrafisz zrobić to **bez użycia żadnych pętli/comprehensions**? 

Przypomnienie, dla dwóch wektorów: $x$ i $y$ $$d_{e}(x,y) = \| x - y \|^2_2 = \sum_{i=1}^d (x_i - y_i)^2 $$

In [6]:
points = np.random.default_rng(0).random((10, 5))


def pairwise_squared_distances(points):
    points = np.asarray(points, dtype=float)
    return np.sum((points[:, None, :] - points[None, :, :]) ** 2, axis=-1)


distances = pairwise_squared_distances(points)
assert distances.shape == (10, 10)
assert np.allclose(distances, distances.T) and np.allclose(np.diag(distances), 0)
distances

array([[0.        , 0.95626093, 0.77715863, 1.04815395, 1.22496952,
        1.95072524, 0.60556809, 1.92269229, 0.59424133, 0.40988719],
       [0.95626093, 0.        , 0.69277865, 0.96652022, 1.13128232,
        0.59678799, 0.65052972, 0.88888833, 0.55346478, 0.37754337],
       [0.77715863, 0.69277865, 0.        , 1.41508384, 1.05943697,
        1.62230395, 1.20204743, 1.43915965, 0.43764519, 0.12390988],
       [1.04815395, 0.96652022, 1.41508384, 0.        , 0.74223096,
        0.45503202, 0.84144309, 0.68821173, 0.45494876, 1.13647187],
       [1.22496952, 1.13128232, 1.05943697, 0.74223096, 0.        ,
        0.9872121 , 0.80621708, 0.40673902, 0.48566729, 1.12016592],
       [1.95072524, 0.59678799, 1.62230395, 0.45503202, 0.9872121 ,
        0.        , 1.19511673, 0.4226464 , 0.82862831, 1.37255878],
       [0.60556809, 0.65052972, 1.20204743, 0.84144309, 0.80621708,
        1.19511673, 0.        , 0.79479326, 0.39404995, 0.80593326],
       [1.92269229, 0.88888833, 1.4391596

### Zad 6.

Napisz funkcje do "wybielania" zbioru punktow, tj. aplikowania następującej transformacji (ponownie, zrób to bez użycia pętli etc.)
$$
\mathrm{white}(X)_{ij} = \frac{X_{ij} - \mu_j}{\sigma_j}
$$
gdzie $\mu_j$ to srednia po $j$ej kolumnie, a $\sigma_j$ to odchylenie standardowe po tej kolumnie, np.

$$
\mathrm{white}\left ( \left [ \begin{matrix} 
0& 1\\
2& 3\\
4& 5\\
6& 7\\
8& 9
\end{matrix} \right ] \right ) = 
\left [ \begin{matrix} 
-1.41421356& -1.41421356\\
-0.70710678& -0.70710678\\
0.        &  0.        \\
0.70710678&  0.70710678\\
1.41421356&  1.41421356
\end{matrix} \right ]
$$

W szczególności średnia po poszczególnych kolumnach nowej macierzy powinna wynosić `0` a odchylenie standardowe `1`.

In [7]:
def whiten(points):
    points = np.asarray(points, dtype=float)
    means = points.mean(axis=0)
    stds = points.std(axis=0)
    if np.any(stds == 0):
        raise ValueError("cannot whiten a constant column")
    return (points - means) / stds


sample = np.arange(10).reshape(5, 2)
white_sample = whiten(sample)
np.testing.assert_allclose(white_sample.mean(axis=0), 0)
np.testing.assert_allclose(white_sample.std(axis=0), 1)
white_sample

array([[-1.41421356, -1.41421356],
       [-0.70710678, -0.70710678],
       [ 0.        ,  0.        ],
       [ 0.70710678,  0.70710678],
       [ 1.41421356,  1.41421356]])

### Zad 7.

Majac dana tablice dowolnie wymiarowa (dla uproszczenia mozesz przyjac ze jedno lub dwu) oraz liczbe `x` znajdz najblizsza wartosc `x` w `A`, tzn. 

$$
\mathrm{closest}(x, A) = \arg\min_{a\in A}| x - a| 
$$

np. 

$$
\mathrm{closest}(1.5, [1, -4, 3]) = 1
$$

Ponownie, czy potrafisz zrobic to bez petli, etc.?


In [8]:
def closest(x, values):
    values = np.asarray(values)
    return values.flat[np.abs(values - x).argmin()]


assert closest(1.5, [1, -4, 3]) == 1
closest(4.2, np.array([[7, 3], [5, -1]]))

np.int64(5)

### Zad 8.

Zaimplementuj proste liczenie wartosci wielomianu o zadanych wspolczynnikach (analogie `np.poly1d`), czyli funkcje

$$
\mathrm{poly}(x, a) = a_0 + a_1 x + a_2 x^2 + \dots + a_k x^k = \sum_{i=1}^k a_i x^i
$$

Czy potrafisz zrobic to jednolinijkowo bez uzycia petli, etc. (oraz oczywiście `np.poly1d`). 
Może Ci się przydać metoda `cumprod`.

In [9]:
def poly(x, coefficients):
    coefficients = np.asarray(coefficients)
    return np.sum(coefficients * np.asarray(x) ** np.arange(coefficients.size))


assert poly(2, [1, 2, 3]) == 17
poly(3, [2, -1, 4])

np.int64(35)

## Obrazki

Do wyświetlania obrazków w notebooku potrzebny będzie `matplotlib`. W PyCharmie można użyć `imshow` z biblioteki `scipy`.

In [10]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

In [11]:
from pathlib import Path

image_path = Path("images/doggo.png")
if image_path.is_file():
    img = mpimg.imread(image_path)
    plt.imshow(img)
    plt.axis("off")
else:
    print("Optional image images/doggo.png not found.")

Optional image images/doggo.png not found.


### Zad 9.

Napisz funckje do przetwarzania obrazu do skali szarości wg wzoru:

$$ 0.2126 \times R + 0.7152 \times  G + 0.0722 \times B $$

In [12]:
from pathlib import Path


def grayscale(image):
    rgb = np.asarray(image)
    if rgb.ndim < 3 or rgb.shape[-1] < 3:
        raise ValueError("expected image with at least three color channels")
    return rgb[..., :3] @ np.array([0.2126, 0.7152, 0.0722])


np.testing.assert_allclose(grayscale([[[1, 0, 0]]]), [[0.2126]])
if Path("images/doggo.png").is_file():
    doggo_gray = grayscale(mpimg.imread("images/doggo.png"))
    plt.imshow(doggo_gray, cmap="gray")
    plt.axis("off")
else:
    print("Optional image images/doggo.png not found.")

Optional image images/doggo.png not found.


### Zadanie 10.

Wyłącznie za pomocą metod numpy napisz kod do interpolacji liniowej pomiędzy parą punktów (dowolnej wymiarowości) wg wzoru:

$$ x_i = \lambda x_1 + (1 - \lambda) x_2$$

Kod powinien od razu generować interpolacje dla wielu wartości $\lambda$, np dla 10 różnych $\lambda \in \{0, 0.1, 0.2, \dots, 1\}$

In [13]:
def interpolate(x1, x2, lambdas):
    weights = np.asarray(lambdas)[..., None]
    return weights * np.asarray(x1) + (1 - weights) * np.asarray(x2)


lambdas = np.linspace(0, 1, 11)
line = interpolate([0, 0], [10, 20], lambdas)
assert line.shape == (11, 2)
np.testing.assert_allclose(line[[0, -1]], [[10, 20], [0, 0]])
line

array([[10., 20.],
       [ 9., 18.],
       [ 8., 16.],
       [ 7., 14.],
       [ 6., 12.],
       [ 5., 10.],
       [ 4.,  8.],
       [ 3.,  6.],
       [ 2.,  4.],
       [ 1.,  2.],
       [ 0.,  0.]])